# Aetherion GamaX1 — Full Colab Pipeline v3

This notebook is intentionally **validation-first**.

It does not jump straight into expensive training. The order is:

**environment → repository/package integrity → Drive/data integrity → corpus schema → tokenizer → layers → bulk BPE cache → corpus statistics → capacity planning → real-data smoke training → checkpoint/resume → full training → generation → controlled dense comparison**

Current four sources:

1. `books_cleaned_v1` — prose
2. `Math_Reasoning/train/books` — user/assistant
3. `Conversations-200k_clean` — user/assistant
4. `QnA` — JSONL user/assistant Q&A

The notebook also handles the current repository situation where source files may be flat (`train.py`, `model.py`, ...) while the code itself uses package-relative imports (`from .model import ...`).


**v3 changes (bug fixes only, no new cells/behavior added):**
- Cell 24 (`Q&A` source-format check) now passes as-is -- `bulk_corpus.py` already defines a 4th `Q&A` source with proper `.jsonl` support.
- Cell 50 (`compare_dense`) previously passed `--data_dir`/`--bulk_cache_dir`/`--out_dir`/`--max_steps`/`--no_amp`, none of which exist in `compare_dense.py`'s actual CLI -- it would have failed with "unrecognized arguments" the moment it was uncommented. Fixed to use `compare_dense.py`'s real flags (`--data` pointing at one real book file, `--steps`) instead.
- Not a notebook change, but required before Cell 28 (build/load bulk cache) can run without crashing: `bulk_corpus.py` had a duplicate `_file_stat()` definition whose second copy silently shadowed the first and dropped the `"fingerprint"` key that 3 other places in the file still read -- this crashed with `KeyError: 'fingerprint'` on the very first file encoded, and separately, the encode-checkpoint block referenced `interval_elapsed`/`files_per_sec`/`eta_sec` before they were computed, crashing with `UnboundLocalError` on the first checkpoint of any batch. Both are fixed in the `bulk_corpus.py` delivered alongside this notebook -- copy it into your repo before running Cell 28.

## 0. Configuration


In [ ]:
from pathlib import Path
import os

# ---------- Project ----------
REPO_DIR = Path("/content/gamax1_project")

# Leave empty if the repo already exists in /content or will be uploaded below.
# Put your real GitHub URL here only if you want this notebook to clone it.
REPO_URL = ""

# ---------- Drive ----------
DRIVE_ROOT = Path("/content/drive/MyDrive/Aetherion_GamaX1")
DATA_ROOT = DRIVE_ROOT / "data"

BOOKS_DIR = DATA_ROOT / "books_cleaned_v1"
MATH_DIR = DATA_ROOT / "Math_Reasoning" / "train" / "books"
CONVERSATIONS_DIR = DATA_ROOT / "Conversations-200k_clean"
QNA_DIR = DATA_ROOT / "QnA"

CACHE_DIR = DRIVE_ROOT / "cache" / "gamax1_bulk"
RUN_ROOT = DRIVE_ROOT / "runs" / "gamax1"

# ---------- Corpus / tokenizer ----------
BPE_VOCAB_SIZE = 16_000
BPE_SAMPLE_CHARS = 3_000_000
REBUILD_BULK_CACHE = False

# ---------- Real-data smoke test ----------
SMOKE_STEPS = 5
SMOKE_D_MODEL = 64
SMOKE_HEADS = 4
SMOKE_LAYERS = 1
SMOKE_FEATURES = 128
SMOKE_BLOCK = 64
SMOKE_BATCH = 2

# ---------- Full training ----------
D_MODEL = 256
N_HEADS = 4
N_LAYERS = 4
N_FEATURES = 1024
BLOCK_SIZE = 256
BATCH_SIZE = 8
LR = 3e-4
WEIGHT_DECAY = 0.01
DROPOUT = 0.1
MAX_STEPS = 2_000
WARMUP_STEPS = 200
EVAL_INTERVAL = 200
EVAL_BATCHES = 10
CHECKPOINT_INTERVAL = 200
VALIDATION_SPLIT = "tail"
VALIDATION_SPLIT_SEED = 42
DEVICE = "cuda"
NO_AMP = False
AUTO_SIZE_MODEL = False

# Safety: full training is opt-in inside the notebook.
RUN_FULL_TRAINING = False

print("Configuration loaded.")


## 1. Mount Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. Verify the four real data sources


In [ ]:
from collections import Counter

sources = {
    "books_cleaned_v1": BOOKS_DIR,
    "Math_Reasoning_train": MATH_DIR,
    "Conversations-200k_clean": CONVERSATIONS_DIR,
    "Q&A": QNA_DIR,
}

total_files = 0
for name, root in sources.items():
    print(f"\n### {name}")
    print("path   :", root)
    print("exists :", root.exists())
    if not root.exists():
        continue
    files = [p for p in root.rglob("*") if p.is_file()]
    counts = Counter(p.suffix.lower() for p in files)
    print("files  :", len(files))
    print("formats:", dict(counts))
    print("sample :", [p.name for p in files[:5]])
    total_files += len(files)

print("\nTOTAL FILES:", total_files)

assert BOOKS_DIR.exists(), BOOKS_DIR
assert MATH_DIR.exists(), MATH_DIR
assert CONVERSATIONS_DIR.exists(), CONVERSATIONS_DIR
assert QNA_DIR.exists(), QNA_DIR


## 3. Check data quality before tokenization


In [ ]:
import json

def inspect_text_file(path, min_chars=1):
    try:
        text = path.read_text(encoding="utf-8", errors="strict")
        return len(text), None
    except Exception as e:
        return 0, repr(e)

def inspect_jsonl_file(path):
    records = 0
    bad = 0
    try:
        with path.open("r", encoding="utf-8", errors="strict") as f:
            for line_no, line in enumerate(f, 1):
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                    if not isinstance(obj, dict):
                        bad += 1
                    else:
                        records += 1
                except Exception:
                    bad += 1
        return records, bad, None
    except Exception as e:
        return records, bad, repr(e)

quality = {}
for name, root in sources.items():
    files = [p for p in root.rglob("*") if p.is_file()]
    empty = []
    unreadable = []
    records = 0
    bad_jsonl = 0

    for p in files:
        if p.suffix.lower() == ".jsonl":
            r, b, err = inspect_jsonl_file(p)
            records += r
            bad_jsonl += b
            if err:
                unreadable.append((str(p), err))
        else:
            n, err = inspect_text_file(p)
            if err:
                unreadable.append((str(p), err))
            elif n == 0:
                empty.append(str(p))

    quality[name] = {
        "files": len(files),
        "empty_text_files": len(empty),
        "unreadable_files": len(unreadable),
        "jsonl_records": records,
        "bad_jsonl_records": bad_jsonl,
    }

for name, q in quality.items():
    print(name, q)

print("\nThis is an integrity check only; it does not silently delete or rewrite data.")


## 4. Find or obtain the repository


In [ ]:
import subprocess, shutil
from pathlib import Path

# If /content/gamax1_project already exists, use it.
# Otherwise clone if REPO_URL is set.
# Otherwise upload a ZIP containing the project.

if not REPO_DIR.exists():
    if REPO_URL.strip():
        print("Cloning repository...")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        from google.colab import files
        print("Repository not found.")
        print("Upload the GamaX1 project ZIP now.")
        uploaded = files.upload()
        zip_files = [Path("/content") / name for name in uploaded if name.lower().endswith(".zip")]
        if not zip_files:
            raise RuntimeError("No .zip project file was uploaded.")
        import zipfile
        with zipfile.ZipFile(zip_files[0]) as z:
            z.extractall("/content")

        # Try common extracted names.
        candidates = [
            Path("/content/gamax1_project"),
            Path("/content/GamaX1_Aetherion"),
            Path("/content/gamax1"),
        ]
        found = next((p for p in candidates if p.exists()), None)
        if found is None:
            py_hits = list(Path("/content").glob("**/train.py"))
            if not py_hits:
                raise RuntimeError("Could not locate train.py after ZIP extraction.")
            found = py_hits[0].parent
        REPO_DIR = found

print("REPO_DIR:", REPO_DIR.resolve())
print("Repository contents:")
for p in sorted(REPO_DIR.iterdir()):
    print(" ", p.name, "/" if p.is_dir() else "")


## 5. Normalize flat repo into an importable `gamax1` package


In [ ]:
import sys, shutil
from pathlib import Path

# The current source uses relative imports such as `from .model import ...`.
# Therefore `python train.py` / `import train` is not the correct execution mode.
# If the repository is already packaged, keep it. If it is flat, create a
# package mirror inside the Colab runtime.

PACKAGE_DIR = REPO_DIR / "gamax1"

required = [
    "tokenizer.py", "layers.py", "model.py", "train.py",
    "bulk_corpus.py", "instruction_data.py", "generate.py", "compare_dense.py"
]

if PACKAGE_DIR.exists() and (PACKAGE_DIR / "__init__.py").exists():
    print("Existing gamax1 package detected.")
else:
    PACKAGE_DIR.mkdir(exist_ok=True)
    for filename in required:
        src = REPO_DIR / filename
        if not src.exists():
            raise FileNotFoundError(f"Missing required source: {src}")
        shutil.copy2(src, PACKAGE_DIR / filename)
    (PACKAGE_DIR / "__init__.py").write_text(
        '"""Aetherion GamaX1 runtime package."""\n',
        encoding="utf-8",
    )
    print("Flat repository normalized into:", PACKAGE_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Package files:")
for p in sorted(PACKAGE_DIR.iterdir()):
    print(" ", p.name)


## 6. Check Python / GPU without modifying Colab's PyTorch


In [ ]:
import sys, torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: CUDA is unavailable; training will be very slow on CPU.")

# Do not `pip install -U torch` in this notebook.
# Colab's CUDA-compatible PyTorch build should be left intact.


## 7. Import the packaged code


In [ ]:
import gamax1
from gamax1 import tokenizer, layers, model, train, bulk_corpus, instruction_data

print("gamax1 package import: PASS")
print("tokenizer:", tokenizer.__file__)
print("train    :", train.__file__)
print("bulk     :", bulk_corpus.__file__)


## 8. API/integrity checks


In [ ]:
import inspect
from gamax1.tokenizer import BPETokenizer, CharTokenizer
from gamax1.layers import (
    build_hex_neighbor_table,
    DynamicSparsityController,
    ProbationaryMemoryTracker,
    SparseSuperpositionLinear,
)
from gamax1.instruction_data import encode_example

required_symbols = [
    BPETokenizer, CharTokenizer, build_hex_neighbor_table,
    DynamicSparsityController, ProbationaryMemoryTracker,
    SparseSuperpositionLinear, encode_example
]
assert all(x is not None for x in required_symbols)

# Detect the specific historical CharTokenizer contamination bug.
char_source = inspect.getsource(CharTokenizer.encode)
bad_names = ["eos_id", "user_id", "assistant_id", "_merge_trie", "byte_to_id"]
bad_hits = [x for x in bad_names if x in char_source]

if bad_hits:
    print("WARNING: CharTokenizer.encode still contains BPE-only references:", bad_hits)
    print("BPE training can still be tested separately, but the tokenizer file should be fixed before using char mode.")
else:
    print("CharTokenizer implementation looks independent of BPE internals.")

print("Required APIs: PASS")


## 9. BPE tokenizer smoke test


In [ ]:
from gamax1.tokenizer import BPETokenizer

sample = "Aetherion is a small language-model research project. Numbers: 12345."
bpe = BPETokenizer(sample, vocab_size=512)

ids = bpe.encode(sample)
decoded = bpe.decode(ids)

print("vocab size:", bpe.vocab_size)
print("tokens:", len(ids))
print("decoded:", decoded)
assert decoded == sample
print("BPE round-trip: PASS")


## 10. Layers smoke test


In [ ]:
import torch
from gamax1.layers import (
    build_hex_neighbor_table,
    DynamicSparsityController,
    ProbationaryMemoryTracker,
    SparseSuperpositionLinear,
)

torch.manual_seed(42)

n_features = 32
d_model = 32
x = torch.randn(2, 8, d_model)

neighbors = build_hex_neighbor_table(n_features)
assert neighbors.shape == (n_features, 6)
assert neighbors.dtype == torch.long
assert int(neighbors.min()) >= 0 and int(neighbors.max()) < n_features

ctrl = DynamicSparsityController(16, 4, 32, trend_window=10, patience=20)
for i in range(40):
    ctrl.step(1.0 - 0.001 * i)
assert 4 <= ctrl.k <= 32

ptm = ProbationaryMemoryTracker(n_features)
mask = torch.zeros(16, n_features, dtype=torch.bool)
mask[:, :2] = True
for _ in range(4):
    ptm.update(mask)

layer = SparseSuperpositionLinear(d_model, n_features)
y = layer(x, k=12)
y.square().mean().backward()

assert y.shape == x.shape
assert layer.last_active_mask.shape == (16, n_features)

print("neighbor table:", neighbors.shape)
print("controller k:", ctrl.k)
print("active mask:", layer.last_active_mask.shape)
print("LAYERS SMOKE TEST: PASS")


## 11. Inspect source-format assumptions


In [ ]:
from gamax1.bulk_corpus import DEFAULT_SOURCE_DIRS, DEFAULT_SOURCE_FORMATS

print("Default source directories:")
for k, v in DEFAULT_SOURCE_DIRS.items():
    print(f"  {k}: {v}")

print("\nDefault source formats:")
for k, v in DEFAULT_SOURCE_FORMATS.items():
    print(f"  {k}: {v}")

assert set(DEFAULT_SOURCE_FORMATS) >= set(sources)
assert DEFAULT_SOURCE_FORMATS["books_cleaned_v1"] == "prose"
assert DEFAULT_SOURCE_FORMATS["Math_Reasoning_train"] == "user_assistant"
assert DEFAULT_SOURCE_FORMATS["Conversations-200k_clean"] == "user_assistant"
assert DEFAULT_SOURCE_FORMATS["Q&A"] == "user_assistant"

print("\nSource-format mapping: PASS")


## 12. Preview representative files


In [ ]:
import json

def preview(path, n=700):
    print("\nFILE:", path)
    if path.suffix.lower() == ".jsonl":
        with path.open("r", encoding="utf-8") as f:
            for _ in range(2):
                line = f.readline()
                if not line:
                    break
                print(line[:n])
    else:
        print(path.read_text(encoding="utf-8", errors="replace")[:n])

for name, root in sources.items():
    files = [p for p in root.rglob("*") if p.is_file()]
    if files:
        preview(files[0])

print("\nRepresentative preview complete.")


## 13. Build/load persistent bulk BPE cache


In [ ]:
from gamax1.bulk_corpus import build_or_load_bulk_tokens

CACHE_DIR.mkdir(parents=True, exist_ok=True)

tok, store, metadata = build_or_load_bulk_tokens(
    str(DATA_ROOT),
    str(CACHE_DIR),
    bpe_vocab_size=BPE_VOCAB_SIZE,
    bpe_sample_chars=BPE_SAMPLE_CHARS,
    tokenizer=None,
    rebuild=REBUILD_BULK_CACHE,
)

print("Bulk cache ready.")
print("Tokenizer vocab:", tok.vocab_size)
print("Token store:", type(store).__name__)
print("Metadata keys:", sorted(metadata.keys()))
print("File count:", metadata.get("file_count"))
print("Token count:", metadata.get("token_count"))


## 14. Validate cache identity and token ranges


In [ ]:
import json
from pathlib import Path

meta_path = CACHE_DIR / "metadata.json"
token_path = CACHE_DIR / "tokens.int32.bin"

assert meta_path.exists(), meta_path
assert token_path.exists(), token_path

cache_meta = json.loads(meta_path.read_text(encoding="utf-8"))

print("cache metadata:")
for k in ["corpus_format_version", "file_count", "token_count", "vocab_size", "bpe_vocab_size"]:
    if k in cache_meta:
        print(f"  {k}: {cache_meta[k]}")

print("token file MB:", token_path.stat().st_size / 1024**2)

# BulkStore exposes an int32 memory map as `.tensor`.
data = store.tensor
print("tensor dtype:", data.dtype)
print("tensor length:", len(data))

assert str(data.dtype).endswith("int32")
assert len(data) > 0
assert int(data.min()) >= 0
assert int(data.max()) < tok.vocab_size

print("CACHE INTEGRITY: PASS")


## 15. Per-source corpus statistics


In [ ]:
# The metadata contains the source/file index used by the incremental cache.
# Print whatever source-level information this version exposes.

for key in ["source_stats", "sources", "source_token_counts", "file_count", "token_count"]:
    if key in metadata:
        print(f"\n{key}:")
        print(metadata[key])

print("\nIf source-level counts are available above, use them to check that")
print("books, math, conversations, and Q&A all contributed before training.")


## 16. Special-token and instruction-format validation


In [ ]:
specials = ["eos_id", "user_id", "assistant_id", "pad_id", "think_id", "think_end_id"]
for name in specials:
    print(name, getattr(tok, name, None))

assert tok.eos_id is not None
assert tok.user_id is not None
assert tok.assistant_id is not None

sample = "<|user|>What is Aetherion?<|assistant|>Aetherion is a research project.<|eos|>"
ids = tok.encode(sample)
decoded = tok.decode(ids)
print("\nRound-trip:")
print(decoded)

ids2, mask = encode_example(
    tok,
    "What is Aetherion?",
    "Aetherion is a research project.",
    max_len=128,
)

assert ids2[0] == tok.user_id
assert tok.assistant_id in ids2
assert ids2[-1] == tok.eos_id
assert len(ids2) == len(mask)

print("\nInstruction IDs:", len(ids2))
print("Assistant position:", ids2.index(tok.assistant_id))
print("EOS position:", len(ids2)-1)
print("INSTRUCTION FORMAT: PASS")


## 17. Capacity planning BEFORE training


In [ ]:
from gamax1.train import _parameter_count_for_config, tokens_per_parameter, auto_size_model

token_count = int(len(store.tensor))

param_count = _parameter_count_for_config(
    tok.vocab_size,
    BLOCK_SIZE,
    D_MODEL,
    N_HEADS,
    N_LAYERS,
    N_FEATURES,
)

ratio = tokens_per_parameter(token_count, param_count)

print(f"Corpus tokens     : {token_count:,}")
print(f"Vocab             : {tok.vocab_size:,}")
print(f"Model parameters  : {param_count:,}")
print(f"Tokens/parameter  : {ratio:.2f}")

if ratio < 10:
    print("WARNING: corpus/model ratio is below the training heuristic floor.")
elif ratio < 40:
    print("NOTICE: corpus is above the minimum heuristic but below the 40 tok/param target.")
else:
    print("Capacity check: comfortably above the configured 40 tok/param target.")

if AUTO_SIZE_MODEL:
    plan = auto_size_model(
        token_count,
        tok.vocab_size,
        BLOCK_SIZE,
        N_HEADS,
        target_tokens_per_param=40.0,
        min_tokens_per_param=10.0,
    )
    print("\nAUTO-SIZE PLAN:")
    print(plan)


## 18. Real-data smoke training — mandatory gate


In [ ]:
import subprocess, sys
from pathlib import Path

SMOKE_OUT = RUN_ROOT / "smoke_real_data"
SMOKE_OUT.mkdir(parents=True, exist_ok=True)

smoke_cmd = [
    sys.executable, "-m", "gamax1.train",
    "--data_dir", str(DATA_ROOT),
    "--bulk_cache_dir", str(CACHE_DIR),
    "--out_dir", str(SMOKE_OUT),
    "--d_model", str(SMOKE_D_MODEL),
    "--n_heads", str(SMOKE_HEADS),
    "--n_layers", str(SMOKE_LAYERS),
    "--n_features", str(SMOKE_FEATURES),
    "--block_size", str(SMOKE_BLOCK),
    "--batch_size", str(SMOKE_BATCH),
    "--lr", "3e-4",
    "--max_steps", str(SMOKE_STEPS),
    "--warmup_steps", "1",
    "--eval_interval", str(SMOKE_STEPS),
    "--eval_batches", "2",
    "--checkpoint_interval", str(SMOKE_STEPS),
    "--validation_split", VALIDATION_SPLIT,
    "--validation_split_seed", str(VALIDATION_SPLIT_SEED),
    "--tokenizer", "bpe",
    "--bpe_vocab_size", str(BPE_VOCAB_SIZE),
    "--bpe_sample_chars", str(BPE_SAMPLE_CHARS),
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
]
if NO_AMP:
    smoke_cmd.append("--no_amp")

print(" ".join(smoke_cmd))
subprocess.run(smoke_cmd, cwd=str(REPO_DIR), check=True)

assert (SMOKE_OUT / "gamax1.pt").exists()
assert (SMOKE_OUT / "gamax1_latest.pt").exists()

print("REAL-DATA SMOKE TRAINING: PASS")


## 19. Verify smoke checkpoint can resume


In [ ]:
latest = SMOKE_OUT / "gamax1_latest.pt"
resume_out = RUN_ROOT / "resume_test"
resume_out.mkdir(parents=True, exist_ok=True)

resume_cmd = [
    sys.executable, "-m", "gamax1.train",
    "--data_dir", str(DATA_ROOT),
    "--bulk_cache_dir", str(CACHE_DIR),
    "--out_dir", str(resume_out),
    "--resume_from", str(latest),
    "--max_steps", str(SMOKE_STEPS + 2),
    "--eval_interval", "2",
    "--eval_batches", "1",
    "--checkpoint_interval", "2",
    "--tokenizer", "bpe",
    "--bpe_vocab_size", str(BPE_VOCAB_SIZE),
    "--bpe_sample_chars", str(BPE_SAMPLE_CHARS),
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
]
if NO_AMP:
    resume_cmd.append("--no_amp")

print(" ".join(resume_cmd))
subprocess.run(resume_cmd, cwd=str(REPO_DIR), check=True)

print("RESUME TEST: PASS")


## 20. Full training — opt-in


In [ ]:
if not RUN_FULL_TRAINING:
    print("RUN_FULL_TRAINING=False — full training skipped.")
    print("Set RUN_FULL_TRAINING=True and rerun this cell after all gates pass.")
else:
    FULL_OUT = RUN_ROOT / "full"
    FULL_OUT.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable, "-m", "gamax1.train",
        "--data_dir", str(DATA_ROOT),
        "--bulk_cache_dir", str(CACHE_DIR),
        "--out_dir", str(FULL_OUT),
        "--d_model", str(D_MODEL),
        "--n_heads", str(N_HEADS),
        "--n_layers", str(N_LAYERS),
        "--n_features", str(N_FEATURES),
        "--block_size", str(BLOCK_SIZE),
        "--batch_size", str(BATCH_SIZE),
        "--lr", str(LR),
        "--weight_decay", str(WEIGHT_DECAY),
        "--dropout", str(DROPOUT),
        "--max_steps", str(MAX_STEPS),
        "--warmup_steps", str(WARMUP_STEPS),
        "--eval_interval", str(EVAL_INTERVAL),
        "--eval_batches", str(EVAL_BATCHES),
        "--checkpoint_interval", str(CHECKPOINT_INTERVAL),
        "--validation_split", VALIDATION_SPLIT,
        "--validation_split_seed", str(VALIDATION_SPLIT_SEED),
        "--tokenizer", "bpe",
        "--bpe_vocab_size", str(BPE_VOCAB_SIZE),
        "--bpe_sample_chars", str(BPE_SAMPLE_CHARS),
        "--device", DEVICE if DEVICE == "cuda" and torch.cuda.is_available() else "cpu",
    ]

    if AUTO_SIZE_MODEL:
        cmd.append("--auto_size_model")
    if NO_AMP:
        cmd.append("--no_amp")

    print(" ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


## 21. Resume full training


In [ ]:
FULL_OUT = RUN_ROOT / "full"
latest_full = FULL_OUT / "gamax1_latest.pt"

if not latest_full.exists():
    print("No full-run checkpoint exists yet.")
    print("Run the full-training cell first.")
else:
    cmd = [
        sys.executable, "-m", "gamax1.train",
        "--data_dir", str(DATA_ROOT),
        "--bulk_cache_dir", str(CACHE_DIR),
        "--out_dir", str(FULL_OUT),
        "--resume_from", str(latest_full),
        "--max_steps", str(MAX_STEPS),
        "--eval_interval", str(EVAL_INTERVAL),
        "--eval_batches", str(EVAL_BATCHES),
        "--checkpoint_interval", str(CHECKPOINT_INTERVAL),
        "--tokenizer", "bpe",
        "--bpe_vocab_size", str(BPE_VOCAB_SIZE),
        "--bpe_sample_chars", str(BPE_SAMPLE_CHARS),
        "--device", DEVICE if DEVICE == "cuda" and torch.cuda.is_available() else "cpu",
    ]
    if NO_AMP:
        cmd.append("--no_amp")
    subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


## 22. Generation test


In [ ]:
CKPT = RUN_ROOT / "full" / "gamax1.pt"

if not CKPT.exists():
    print("Full checkpoint not found:", CKPT)
else:
    cmd = [
        sys.executable, "-m", "gamax1.generate",
        "--ckpt", str(CKPT),
        "--prompt", "Aetherion is",
        "--max_new_tokens", "80",
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
    ]
    subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


## 23. Chat generation test


In [ ]:
if CKPT.exists():
    cmd = [
        sys.executable, "-m", "gamax1.generate",
        "--ckpt", str(CKPT),
        "--chat",
        "--max_new_tokens", "80",
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
    ]
    subprocess.run(cmd, cwd=str(REPO_DIR), check=True)
else:
    print("Skip: no full checkpoint yet.")


## 24. Controlled sparse-vs-dense comparison


In [ ]:
COMPARE_OUT = RUN_ROOT / "compare_dense"
COMPARE_OUT.mkdir(parents=True, exist_ok=True)  # currently unused: compare_dense.py only prints, saves nothing

# compare_dense.py is a standalone, single-flat-file mechanism test (sparse vs
# dense FFN) -- it does NOT use the bulk_corpus.py multi-source/.jsonl pipeline
# the rest of this notebook uses. Its real CLI only supports:
#   --data --tokenizer --bpe_vocab_size --steps --d_model --n_heads --n_layers
#   --n_features --block_size --batch_size --lr --dropout --eval_batches --device
# (no --data_dir / --bulk_cache_dir / --out_dir / --max_steps / --no_amp).
# It wants ONE plain-text file, so we point it at a real book file instead of
# the multi-source corpus.
compare_sample_files = [p for p in BOOKS_DIR.rglob("*.txt") if p.is_file()]
if not compare_sample_files:
    raise FileNotFoundError(
        f"No .txt files found under {BOOKS_DIR} for compare_dense's --data. "
        "This experiment needs one representative plain-text file."
    )
compare_data_file = compare_sample_files[0]

cmd = [
    sys.executable, "-m", "gamax1.compare_dense",
    "--data", str(compare_data_file),
    "--d_model", str(D_MODEL),
    "--n_heads", str(N_HEADS),
    "--n_layers", str(N_LAYERS),
    "--n_features", str(N_FEATURES),
    "--block_size", str(BLOCK_SIZE),
    "--batch_size", str(BATCH_SIZE),
    "--steps", "200",
    "--eval_batches", str(EVAL_BATCHES),
    "--tokenizer", "bpe",
    "--bpe_vocab_size", str(BPE_VOCAB_SIZE),
    "--device", DEVICE if DEVICE == "cuda" and torch.cuda.is_available() else "cpu",
]

print("This is an optional experiment, not the first training gate.")
print("Comparison data file:", compare_data_file)
print(" ".join(cmd))
# Uncomment to execute:
# subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


## 25. Final artifacts and environment report


In [ ]:
from pathlib import Path

print("### Environment")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n### Cache")
if CACHE_DIR.exists():
    for p in sorted(CACHE_DIR.glob("*")):
        if p.is_file():
            print(p.name, f"{p.stat().st_size/1024/1024:.2f} MB")

print("\n### Runs")
if RUN_ROOT.exists():
    for p in sorted(RUN_ROOT.rglob("*")):
        if p.is_file():
            print(p.relative_to(RUN_ROOT), f"{p.stat().st_size/1024/1024:.2f} MB")

print("\nPipeline completed up to the cells you explicitly ran.")


## Decision gates

Before serious training, all of these should be green:

- [ ] All four Drive sources exist.
- [ ] File formats match expectations, including **Q&A `.jsonl`**.
- [ ] No unexpected unreadable corpus files.
- [ ] Repository/package imports succeed.
- [ ] BPE round-trip succeeds.
- [ ] Layers smoke test succeeds.
- [ ] Bulk cache has a valid identity and token range.
- [ ] All four source categories contribute data.
- [ ] Tokens/parameter capacity is understood.
- [ ] Real-data smoke training succeeds.
- [ ] Checkpoint resume succeeds.
- [ ] Only then start the expensive full run.

**Important:** generation quality from a 5-step smoke test is not a quality benchmark. It only proves that the end-to-end technical path works.
